In [ ]:
# open its related files and executes its files:

from pathlib import Path
import pandas as pd

data_folder = Path("data") / "merged_data"

# If this notebook is inside the notebooks folder, use the parent folder.
if not data_folder.exists():
    data_folder = Path("..") / "data" / "merged_data"

print("Data folder:", data_folder.resolve())

tests = pd.read_csv(data_folder / "booking_tests.csv", dtype=str)
bookings = pd.read_csv(data_folder / "bookings_combined.csv", dtype=str)
plan = pd.read_csv(data_folder / "plan_2026_H1.csv", dtype=str)

print("Test rows:", len(tests))
print("Booking rows:", len(bookings))
print("Plan rows:", len(plan))

In [ ]:
# check each ID has consistent data:

metadata_columns = ["city", "channel", "source_system", "source_month"]

conflicts = (
    bookings.groupby("booking_id")[metadata_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
    .sum()
)

print("Booking IDs with conflicting metadata:", conflicts)

if conflicts > 0:
    raise ValueError("Review conflicting booking metadata before continuing.")

In [ ]:
# Add booking channel to test rows and standardize city names:

booking_info = bookings[
    ["booking_id", "city", "channel"]
].drop_duplicates("booking_id")

tests = tests.merge(
    booking_info[["booking_id", "channel"]],
    on="booking_id",
    how="left",
    validate="many_to_one"
)

tests["channel"] = (
    tests["channel"]
    .fillna("Unmatched")
    .str.strip()
    .str.lower()
    .str.title()
)

tests["city"] = tests["source_city"].str.strip()
tests["city"] = tests["city"].replace({
    "Bengaluru": "Bangalore",
    "Bangalore": "Bangalore"
})

print("Test rows without a booking channel:", tests["channel"].eq("Unmatched").sum())


In [ ]:
# Count H1 2025 and H1 2026 test rows; calculate actual growth, planned growth, and the percentage-point gap:

h1_months = [
    f"{year}-{month:02d}"
    for year in [2025, 2026]
    for month in range(1, 7)
]

h1_tests = tests[tests["source_month"].isin(h1_months)].copy()
h1_tests["year"] = h1_tests["source_month"].str[:4]

tests_2025 = (h1_tests["year"] == "2025").sum()
tests_2026 = (h1_tests["year"] == "2026").sum()

actual_growth = (tests_2026 / tests_2025 - 1) * 100

plan_months = [column for column in plan.columns if column.startswith("2026-")]
plan[plan_months] = plan[plan_months].apply(pd.to_numeric, errors="coerce")
plan["city"] = plan["city"].str.strip().replace({
    "Bengaluru": "Bangalore",
    "Bangalore": "Bangalore"
})

plan["planned_tests"] = plan[plan_months].sum(axis=1)
plan_total = plan["planned_tests"].sum()
planned_growth = (plan_total / tests_2025 - 1) * 100
gap_points = planned_growth - actual_growth

print(f"H1 2025 tests: {tests_2025:,}")
print(f"H1 2026 tests: {tests_2026:,}")
print(f"Actual growth: {actual_growth:.2f}%")
print(f"Planned growth: {planned_growth:.2f}%")
print(f"Gap: {gap_points:.2f} percentage points")



In [ ]:
# Compare test counts and shortfalls by city:

city_counts = (
    h1_tests.groupby(["city", "year"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"2025": "tests_2025", "2026": "tests_2026"})
)

city_summary = city_counts.merge(
    plan[["city", "planned_tests"]],
    on="city",
    how="outer"
).fillna(0)

city_summary["tests_below_plan"] = (
    city_summary["planned_tests"] - city_summary["tests_2026"]
)

city_summary["actual_growth_percent"] = (
    (city_summary["tests_2026"] / city_summary["tests_2025"] - 1) * 100
)

city_summary["share_of_plan_gap_percent"] = (
    city_summary["tests_below_plan"]
    / city_summary["tests_below_plan"].sum()
    * 100
)

city_summary = city_summary.sort_values(
    "tests_below_plan",
    ascending=False
)

city_summary.round(2)



In [ ]:
# Compare test growth and contribution by channel.

channel_counts = (
    h1_tests.groupby(["channel", "year"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"2025": "tests_2025", "2026": "tests_2026"})
)

channel_counts["change_in_tests"] = (
    channel_counts["tests_2026"] - channel_counts["tests_2025"]
)

channel_counts["growth_percent"] = (
    (channel_counts["tests_2026"] / channel_counts["tests_2025"] - 1) * 100
)

channel_counts["growth_contribution_points"] = (
    channel_counts["change_in_tests"] / tests_2025 * 100
)

channel_counts.round(2)

In [ ]:
# link each tests to pateient:

booking_people = bookings[["booking_id", "patient_id"]].copy()

booking_people["booking_id"] = (
    booking_people["booking_id"].astype("string").str.strip()
)
booking_people["patient_id"] = (
    booking_people["patient_id"].astype("string").str.strip()
)

booking_people = booking_people.drop_duplicates()

# Each booking should map to one patient.
if booking_people["booking_id"].duplicated().any():
    raise ValueError("Some booking IDs map to more than one patient ID.")

analysis_tests = tests.drop(columns=["patient_id"], errors="ignore").copy()
analysis_tests["booking_id"] = (
    analysis_tests["booking_id"].astype("string").str.strip()
)

analysis_tests = analysis_tests.merge(
    booking_people,
    on="booking_id",
    how="left",
    validate="many_to_one"
)

print("Test rows without a patient ID:", analysis_tests["patient_id"].isna().sum())

In [ ]:
# Select H1 2025 and H1 2026 test rows:

h1_months = [
    f"{year}-{month:02d}"
    for year in [2025, 2026]
    for month in range(1, 7)
]

analysis_tests["source_month"] = (
    analysis_tests["source_month"].astype("string").str.strip()
)

h1_tests = analysis_tests[
    analysis_tests["source_month"].isin(h1_months)
].copy()

h1_tests["year"] = h1_tests["source_month"].str[:4]

missing_ids = (
    h1_tests["booking_id"].isna()
    | h1_tests["patient_id"].isna()
)

print("H1 test rows missing a booking or patient ID:", missing_ids.sum())

# These rows cannot be used to calculate patients and visits.
h1_tests = h1_tests.loc[~missing_ids].copy()

In [ ]:
# Calculate patients, visits, and tests per visit overall:

overall = (
    h1_tests.groupby("year")
    .agg(
        patients=("patient_id", "nunique"),
        visits=("booking_id", "nunique"),
        tests=("booking_id", "size")
    )
    .reset_index()
)

overall["visits_per_patient"] = (
    overall["visits"] / overall["patients"]
)

overall["tests_per_visit"] = (
    overall["tests"] / overall["visits"]
)

overall["reconstructed_tests"] = (
    overall["patients"]
    * overall["visits_per_patient"]
    * overall["tests_per_visit"]
)

overall.round(2)

In [ ]:
# Compare those factors by city and channel:

def summarize_by(data, group_column):
    summary = (
        data.groupby([group_column, "year"], dropna=False)
        .agg(
            patients=("patient_id", "nunique"),
            visits=("booking_id", "nunique"),
            tests=("booking_id", "size")
        )
        .reset_index()
    )

    summary["visits_per_patient"] = (
        summary["visits"] / summary["patients"]
    )
    summary["tests_per_visit"] = (
        summary["tests"] / summary["visits"]
    )

    return summary.round(2)


city_factors = summarize_by(h1_tests, "city")
channel_factors = summarize_by(h1_tests, "channel")

display(city_factors)
display(channel_factors)

In [ ]:
# Recalculate H1 growth and compare patients, visits, and tests per visit.
# Keep only January–June 2025 and 2026 tests.
h1_months = [
    f"{year}-{month:02d}"
    for year in ["2025", "2026"]
    for month in range(1, 7)
]

h1_tests = analysis_tests[
    analysis_tests["source_month"].isin(h1_months)
].copy()

h1_tests["year"] = h1_tests["source_month"].str[:4]

# Every test needs a booking and patient ID for this breakdown.
missing_ids = (
    h1_tests["booking_id"].isna()
    | h1_tests["patient_id"].isna()
).sum()

print("H1 test rows missing a booking or patient ID:", missing_ids)

if missing_ids > 0:
    raise ValueError("Resolve missing booking or patient IDs before continuing.")

# Calculate the three factors for each year.
overall = (
    h1_tests.groupby("year")
    .agg(
        patients=("patient_id", "nunique"),
        visits=("booking_id", "nunique"),
        tests=("booking_id", "size")
    )
    .reset_index()
)

overall["visits_per_patient"] = overall["visits"] / overall["patients"]
overall["tests_per_visit"] = overall["tests"] / overall["visits"]

# Use only the H1 2026 plan columns.
plan_months = [
    f"2026-{month:02d}"
    for month in range(1, 7)
    if f"2026-{month:02d}" in plan.columns
]

if not plan_months:
    raise ValueError("No January–June 2026 plan columns were found.")

plan[plan_months] = plan[plan_months].apply(
    pd.to_numeric, errors="coerce"
)
plan["planned_tests"] = plan[plan_months].sum(axis=1)

years = overall.set_index("year")

tests_2025 = years.loc["2025", "tests"]
tests_2026 = years.loc["2026", "tests"]

actual_growth = (tests_2026 / tests_2025 - 1) * 100
planned_growth = (plan["planned_tests"].sum() / tests_2025 - 1) * 100
gap_points = planned_growth - actual_growth

# Compare each factor between the two H1 periods.
factor_comparison = years[
    ["patients", "visits", "visits_per_patient", "tests_per_visit", "tests"]
].T
factor_comparison.columns = ["H1 2025", "H1 2026"]
factor_comparison["change_percent"] = (
    factor_comparison["H1 2026"] / factor_comparison["H1 2025"] - 1
) * 100

display(factor_comparison.round(2))

print(f"H1 actual test growth: {actual_growth:.2f}%")
print(f"H1 planned growth: {planned_growth:.2f}%")
print(f"Growth gap: {gap_points:.2f} percentage points")

In [ ]:
# Standardize city names in the plan before matching:

plan["city"] = plan["city"].str.strip().replace({
    "Bengaluru": "Bangalore",
    "Bangalore": "Bangalore"
})

# City test counts.
city_counts = (
    h1_tests.groupby(["city", "year"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["2025", "2026"], fill_value=0)
    .reset_index()
    .rename(columns={"2025": "tests_2025", "2026": "tests_2026"})
)

city_summary = city_counts.merge(
    plan.groupby("city", as_index=False)["planned_tests"].sum(),
    on="city",
    how="outer"
).fillna(0)

city_summary["tests_below_plan"] = (
    city_summary["planned_tests"] - city_summary["tests_2026"]
)
city_summary["actual_growth_percent"] = (
    (city_summary["tests_2026"] / city_summary["tests_2025"] - 1) * 100
).where(city_summary["tests_2025"] != 0)

display(city_summary.sort_values(
    "tests_below_plan", ascending=False
).round(2))

# Channel test counts and contribution to overall growth.
channel_summary = (
    h1_tests.groupby(["channel", "year"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["2025", "2026"], fill_value=0)
    .rename(columns={"2025": "tests_2025", "2026": "tests_2026"})
)

channel_summary["change_in_tests"] = (
    channel_summary["tests_2026"] - channel_summary["tests_2025"]
)
channel_summary["growth_percent"] = (
    (channel_summary["tests_2026"] / channel_summary["tests_2025"] - 1) * 100
).where(channel_summary["tests_2025"] != 0)
channel_summary["growth_contribution_points"] = (
    channel_summary["change_in_tests"] / tests_2025 * 100
)

display(channel_summary.round(2))

: 

# 13. Select January–June 2025 and 2026:

tests["source_month"] = tests["source_month"].fillna("").str.strip()

h1_months = [
    f"{year}-{month:02d}"
    for year in ["2025", "2026"]
    for month in range(1, 7)
]

monthly_tests = tests[
    tests["source_month"].isin(h1_months)
].copy()

monthly_tests["year"] = monthly_tests["source_month"].str[:4]
monthly_tests["month"] = monthly_tests["source_month"].str[-2:]

In [ ]:
# 14. Count test lines by month and year:

month_order = ["01", "02", "03", "04", "05", "06"]

monthly_counts = (
    monthly_tests.groupby(["month", "year"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=month_order, columns=["2025", "2026"], fill_value=0)
    .rename(columns={
        "2025": "tests_2025",
        "2026": "tests_2026"
    })
)

monthly_counts.index = [
    "January", "February", "March", "April", "May", "June"
]

monthly_counts


In [ ]:
# 15. Calculate monthly growth and the running gap:

monthly_trend = monthly_counts.copy()

monthly_trend["change_in_tests"] = (
    monthly_trend["tests_2026"] - monthly_trend["tests_2025"]
)

monthly_trend["growth_percent"] = (
    (monthly_trend["tests_2026"] / monthly_trend["tests_2025"] - 1) * 100
).where(monthly_trend["tests_2025"] != 0)

monthly_trend["cumulative_2025"] = monthly_trend["tests_2025"].cumsum()
monthly_trend["cumulative_2026"] = monthly_trend["tests_2026"].cumsum()

monthly_trend["cumulative_gap"] = (
    monthly_trend["cumulative_2026"] - monthly_trend["cumulative_2025"]
)

monthly_trend["growth_change_from_previous_month_pp"] = (
    monthly_trend["growth_percent"].diff()
)

monthly_trend.round(2)

In [ ]:
# 16. Find months when growth slowed:

slower_months = monthly_trend[
    monthly_trend["growth_change_from_previous_month_pp"] < 0
]

print("Months when year-over-year growth slowed:")
display(slower_months.round(2))

print("Monthly comparison:")
display(monthly_trend.round(2))

In [ ]:
# 17. Count test lines by month, city, and channel:

city_channel_month = (
    monthly_tests
    .groupby(["month", "city", "channel", "year"])
    .size()
    .unstack("year", fill_value=0)
    .reindex(columns=["2025", "2026"], fill_value=0)
    .rename(columns={
        "2025": "tests_2025",
        "2026": "tests_2026"
    })
    .reset_index()
)

city_channel_month.head()

In [ ]:
# 18. Calculate the change and growth:

city_channel_month["change_in_tests"] = (
    city_channel_month["tests_2026"]
    - city_channel_month["tests_2025"]
)

city_channel_month["growth_percent"] = (
    (city_channel_month["tests_2026"]
     / city_channel_month["tests_2025"] - 1) * 100
).where(city_channel_month["tests_2025"] != 0)

city_channel_month = city_channel_month.sort_values(
    ["month", "change_in_tests"],
    ascending=[True, False]
)

city_channel_month.round(2)

In [ ]:
# 19. See the biggest increases and declines for each month:

month_names = {
    "01": "January", "02": "February", "03": "March",
    "04": "April", "05": "May", "06": "June"
}

for month in ["01", "02", "03", "04", "05", "06"]:
    month_data = city_channel_month[
        city_channel_month["month"] == month
    ]

    print(f"\n{month_names[month]} — largest increases")
    display(month_data.nlargest(5, "change_in_tests"))

    print(f"{month_names[month]} — largest declines")
    display(month_data.nsmallest(5, "change_in_tests"))

In [ ]:
# 20. Check that each booking maps to one patient:

booking_people = bookings[["booking_id", "patient_id"]].copy()

for column in ["booking_id", "patient_id"]:
    booking_people[column] = (
        booking_people[column].astype("string").str.strip()
    )

booking_people = booking_people.replace("", pd.NA)
booking_people = booking_people.dropna(
    subset=["booking_id", "patient_id"]
).drop_duplicates()

conflicts = (
    booking_people.groupby("booking_id")["patient_id"]
    .nunique()
    .gt(1)
    .sum()
)

print("Booking IDs linked to multiple patient IDs:", conflicts)

if conflicts > 0:
    raise ValueError("Resolve these booking-to-patient conflicts first.")

booking_people = booking_people.drop_duplicates("booking_id")

In [ ]:
# 21. Prepare H1 test lines and attach patient IDs:

h1_months = {
    f"{year}-{month:02d}"
    for year in ["2025", "2026"]
    for month in range(1, 7)
}

driver_tests = tests.drop(columns=["patient_id"], errors="ignore").copy()

driver_tests["booking_id"] = (
    driver_tests["booking_id"].astype("string").str.strip()
)
driver_tests["source_month"] = (
    driver_tests["source_month"].fillna("").str.strip()
)

driver_tests = driver_tests[
    driver_tests["source_month"].isin(h1_months)
].copy()

driver_tests = driver_tests.merge(
    booking_people,
    on="booking_id",
    how="left",
    validate="many_to_one"
)

missing_ids = driver_tests["patient_id"].isna().sum()
print("H1 test lines without a patient ID:", missing_ids)

if missing_ids > 0:
    raise ValueError("Resolve missing patient IDs before calculating drivers.")

driver_tests["year"] = driver_tests["source_month"].str[:4]

In [ ]:
# 22. Calculate the three factors:

drivers = (
    driver_tests.groupby("year")
    .agg(
        patient_count=("patient_id", "nunique"),
        visit_count=("booking_id", "nunique"),
        test_lines=("booking_id", "size")
    )
    .reindex(["2025", "2026"])
)

if drivers.isna().any().any():
    raise ValueError("Both H1 2025 and H1 2026 data are needed.")

drivers["visits_per_patient"] = (
    drivers["visit_count"] / drivers["patient_count"]
)

drivers["tests_per_visit"] = (
    drivers["test_lines"] / drivers["visit_count"]
)

drivers.round(2)

In [ ]:
# 23. Compare the factors year over year:

factor_columns = [
    "patient_count",
    "visits_per_patient",
    "tests_per_visit",
    "test_lines"
]

comparison = pd.DataFrame({
    "H1_2025": drivers.loc["2025", factor_columns],
    "H1_2026": drivers.loc["2026", factor_columns]
})

comparison["change_percent"] = (
    (comparison["H1_2026"] / comparison["H1_2025"] - 1) * 100
)

display(comparison.round(2))

In [ ]:
# 24. Find the month with the lowest year-over-year growth:

monthly_growth = monthly_trend["growth_percent"].dropna()

weakest_month = monthly_growth.idxmin()

month_codes = {
    "January": "01",
    "February": "02",
    "March": "03",
    "April": "04",
    "May": "05",
    "June": "06"
}

weakest_month_code = month_codes[weakest_month]

print("Lowest-growth month:", weakest_month)
print(f"Year-over-year growth: {monthly_growth.loc[weakest_month]:.2f}%")

In [ ]:
# 25. Find the largest city/channel decline in that month:

weak_month_rows = city_channel_month[
    city_channel_month["month"] == weakest_month_code
].copy()

declines = weak_month_rows[
    weak_month_rows["change_in_tests"] < 0
].sort_values("change_in_tests")

if declines.empty:
    print("No city/channel combination declined in this month.")
else:
    print("Largest city/channel decline in", weakest_month)
    display(declines.head(1))

In [ ]:
# 26. Print the finding and recommended follow-up:

h1_2025 = drivers.loc["2025", "test_lines"]
h1_2026 = drivers.loc["2026", "test_lines"]
actual_growth = (h1_2026 / h1_2025 - 1) * 100

print("SUPPORTED FINDING")
print(
    f"Booked test lines changed from {h1_2025:,.0f} in H1 2025 "
    f"to {h1_2026:,.0f} in H1 2026 ({actual_growth:+.2f}%)."
)
print(
    f"{weakest_month} had the lowest year-over-year monthly growth "
    f"({monthly_growth.loc[weakest_month]:+.2f}%)."
)

if not declines.empty:
    row = declines.iloc[0]
    print(
        f"The largest decline that month was {row['city']} / {row['channel']}: "
        f"{row['change_in_tests']:,.0f} test lines."
    )

factor_changes = comparison.loc[
    ["patient_count", "visits_per_patient", "tests_per_visit"],
    "change_percent"
]
print(
    "Factor with the smallest percentage change:",
    f"{factor_changes.idxmin()} ({factor_changes.min():+.2f}%)."
)

print("\nRECOMMENDED FOLLOW-UP")
print(
    "Review the weakest month and its city/channel pattern with Operations. "
    "Check relevant capacity, booking, and test-mix data before deciding on an intervention."
)
print(
    "This analysis identifies where the change occurred; it does not by itself prove its cause."
)
print(
    "Treat patient-count results cautiously until Legacy and Nova patient IDs "
    "are confirmed to map to unique people."
)